# Pandas — Exercises

**Companion to deck 03 (Pandas Basics).** Run cells in order.

Each problem ships:
- a TODO cell — write your code
- an assertion cell — checks your answer
- a hint cell (run only if stuck)
- a solution cell at the bottom (don't peek too early)

Rule: **no Python `for` loop over rows**. One-liners only.

<a href="https://colab.research.google.com/github/Petkub/MachineLearningLab/blob/main/HTMLSlides/decks/03-pandas-basics/exercises.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Setup — load real Titanic

We use the seaborn copy of Titanic — same shape as the Kaggle one, no auth needed.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns

df = sns.load_dataset('titanic')
# rename a couple of columns to match the Kaggle naming
df = df.rename(columns={'pclass': 'Pclass', 'sex': 'Sex', 'age': 'Age',
                        'sibsp': 'SibSp', 'parch': 'Parch', 'fare': 'Fare',
                        'survived': 'Survived', 'embarked': 'Embarked'})
print(df.shape)
df.head()

---
## Problem 01 — what's in this DataFrame?

**Think first.** You don't need a loop. Pandas exposes ready-made attributes for size and types.

Tasks:
1. Save number of rows in `n_rows`.
2. Save number of columns in `n_cols`.
3. Save the list of *numeric* column names in `numeric_cols`.

In [ ]:
# TODO: fill in
n_rows = ...
n_cols = ...
numeric_cols = ...

print(n_rows, n_cols, numeric_cols)

In [ ]:
assert n_rows == 891, f'expected 891 rows, got {n_rows}'
assert n_cols == len(df.columns), 'n_cols mismatch'
assert set(['Age', 'Pclass', 'Fare', 'Survived']).issubset(set(numeric_cols)), 'numeric_cols missing key columns'
print('Q1 ok')

<details><summary>Hint</summary>

Shape is one attribute: `df.shape` returns `(rows, cols)`. For numeric columns: `df.select_dtypes(include='number').columns.tolist()`.
</details>

---
## Problem 02 — pull out rows that match

Find children (`Age < 12`) traveling in third class (`Pclass == 3`).

**Trap.** Python's `and` does *not* work on Series. Use `&` and parens around each clause.

Tasks:
1. Build boolean Series `mask` — True where both conditions hold.
2. Slice rows into `kids3`.
3. Keep only `Name` and `Age` columns from `kids3` → `kids3_short`.

*Note: seaborn version uses `who` instead of `Name`. We'll use the existing `who` column for this exercise.*

In [ ]:
# TODO
mask = ...
kids3 = ...
kids3_short = ...   # keep only ['who', 'Age']

print('count:', mask.sum())
kids3_short.head()

In [ ]:
assert mask.dtype == bool, 'mask must be a boolean Series'
assert mask.sum() > 0, 'mask matches zero rows — check logic'
assert (kids3['Age'] < 12).all(), 'kids3 has rows where Age >= 12'
assert (kids3['Pclass'] == 3).all(), 'kids3 has rows not in Pclass 3'
assert list(kids3_short.columns) == ['who', 'Age'], f'columns wrong: {list(kids3_short.columns)}'
print('Q2 ok')

<details><summary>Hint</summary>

`mask = (df['Age'] < 12) & (df['Pclass'] == 3)`. Then `df[mask]` slices rows. Subset columns: `kids3[['who','Age']]`.
</details>

---
## Problem 03 — patch holes

Column `Age` has missing values. Don't drop rows — fill with the mean of non-missing ages.

Tasks:
1. Save count of missing ages in `n_missing`.
2. Save mean over non-missing values in `avg_age`.
3. Replace NaN in `df['Age']` with `avg_age` (in place — assign back to `df['Age']`).

In [ ]:
# TODO
n_missing = ...
avg_age   = ...
df['Age'] = ...

print('n_missing:', n_missing, '| avg_age:', round(avg_age, 2), '| remaining NaN:', df['Age'].isnull().sum())

In [ ]:
assert n_missing == 177, f'expected 177 missing ages, got {n_missing}'
assert abs(avg_age - 29.7) < 0.5, f'avg_age looks off: {avg_age}'
assert df['Age'].isnull().sum() == 0, 'NaN still present in Age'
print('Q3 ok')

<details><summary>Hint</summary>

`df['Age'].isnull().sum()` counts NaN. `df['Age'].mean()` already skips NaN. Then assign: `df['Age'] = df['Age'].fillna(avg_age)`.
</details>

---
## Problem 04 — did class matter for survival?

Compute survival rate per Pclass.

Tasks:
1. Save Series `rate` — survival rate per Pclass (mean of 0/1 column = rate).
2. Save the safest Pclass (highest rate) in `safest_class`.
3. Save the spread `gap` = max rate − min rate.

In [ ]:
# TODO
rate = ...
safest_class = ...
gap = ...

print(rate)
print('safest:', safest_class, '| gap:', round(gap, 3))

In [ ]:
assert isinstance(rate, pd.Series), 'rate must be a Series'
assert safest_class == 1, f'expected safest_class=1, got {safest_class}'
assert abs(gap - 0.39) < 0.05, f'gap looks off: {gap}'
print('Q4 ok')

<details><summary>Hint</summary>

`df.groupby(KEY)[METRIC].mean()`. Then `rate.idxmax()` for the winning key, `rate.max() - rate.min()` for spread.
</details>

---
## Problem 05 — derived columns

Build a family-size column from `SibSp` + `Parch`. Then count solo travelers.

Tasks:
1. Add column `FamilySize = SibSp + Parch + 1` (passenger themselves counts).
2. Add boolean column `IsAlone = (FamilySize == 1)`.
3. Save `n_alone` = count of alone passengers, and `frac_alone` = fraction.

In [ ]:
# TODO
df['FamilySize'] = ...
df['IsAlone']    = ...
n_alone    = ...
frac_alone = ...

print('alone:', n_alone, '| fraction:', round(frac_alone, 3))
df[['SibSp', 'Parch', 'FamilySize', 'IsAlone']].head()

In [ ]:
assert (df['FamilySize'] >= 1).all(), 'FamilySize should be >= 1 (counts the passenger)'
assert df['IsAlone'].dtype == bool, 'IsAlone should be boolean'
assert n_alone == int(df['IsAlone'].sum()), 'n_alone count mismatch'
assert abs(frac_alone - 0.60) < 0.05, f'frac_alone looks off: {frac_alone}'
print('Q5 ok')

<details><summary>Hint</summary>

`df['FamilySize'] = df['SibSp'] + df['Parch'] + 1`. `df['IsAlone'] = df['FamilySize'] == 1`. `n_alone = df['IsAlone'].sum()`. `frac_alone = n_alone / len(df)`.
</details>

---
## Problem 06 — filter, then split, then compare

Among adult women (`Age >= 18 & Sex == 'female'`), how big is the fare gap between 1st class and the average of 2nd+3rd?

Tasks:
1. Build `aw` — adult women only.
2. From `aw`, `fare_by_class` — Series of mean Fare per Pclass.
3. `fare_gap` = `fare_by_class[1]` − mean of `fare_by_class[[2, 3]]`.

In [ ]:
# TODO
aw = ...
fare_by_class = ...
fare_gap = ...

print(fare_by_class)
print('fare_gap:', round(fare_gap, 2))

In [ ]:
assert (aw['Sex'] == 'female').all(), 'aw contains non-female rows'
assert (aw['Age'] >= 18).all(), 'aw contains rows with Age < 18'
assert fare_gap > 50, f'fare_gap suspiciously small: {fare_gap}'
print('Q6 ok — adult women in 1st class paid roughly', round(fare_by_class.loc[1] / fare_by_class.loc[3], 1), 'times more than 3rd class')

<details><summary>Hint</summary>

`aw = df[(df['Age'] >= 18) & (df['Sex'] == 'female')]`. Then `aw.groupby('Pclass')['Fare'].mean()`. Index in: `fare_by_class.loc[1]` and `fare_by_class.loc[[2, 3]].mean()`.
</details>

---
## Solutions (don't peek too early)

<details><summary>Show all solutions</summary>

```python
# Q1
n_rows, n_cols = df.shape
numeric_cols = df.select_dtypes(include='number').columns.tolist()

# Q2
mask = (df['Age'] < 12) & (df['Pclass'] == 3)
kids3 = df[mask]
kids3_short = kids3[['who', 'Age']]

# Q3
n_missing = df['Age'].isnull().sum()
avg_age   = df['Age'].mean()
df['Age'] = df['Age'].fillna(avg_age)

# Q4
rate = df.groupby('Pclass')['Survived'].mean()
safest_class = rate.idxmax()
gap = rate.max() - rate.min()

# Q5
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['IsAlone']    = df['FamilySize'] == 1
n_alone    = int(df['IsAlone'].sum())
frac_alone = n_alone / len(df)

# Q6
aw = df[(df['Age'] >= 18) & (df['Sex'] == 'female')]
fare_by_class = aw.groupby('Pclass')['Fare'].mean()
fare_gap = fare_by_class.loc[1] - fare_by_class.loc[[2, 3]].mean()
```
</details>

## Recap

Pattern across all 6: **read what's there → filter / group → summarize**. Same shape every time. Memorize the verbs:

| verb | call |
|---|---|
| inspect | `.shape`, `.dtypes`, `.head()`, `.info()` |
| filter | `df[mask]` with boolean Series |
| repair | `.fillna(...)`, `.dropna()` |
| summarize | `.groupby(KEY)[METRIC].agg(...)` |
| derive | `df['new'] = expr` |